# ICML 16_Runs Experiment Analysis

Comprehensive analysis of grokking dynamics across 16 architectural/optimizer configurations with 6 weight decay values (96 total experiments).

## Experiment Factors
- **Modulus**: 97 vs 113
- **Attention**: Softmax vs ReLU
- **LayerNorm**: On vs Off
- **Optimizer**: Adam vs Muon
- **Weight Decay**: 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0


In [ ]:
import os
import sys
import json
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Results directory
RESULTS_DIR = Path('../results')
FIGURES_DIR = Path('./figures')
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Results directory: {RESULTS_DIR.absolute()}")
print(f"Figures directory: {FIGURES_DIR.absolute()}")


## 1. Load All Experiment Results


In [ ]:
def load_experiment(exp_dir):
    """Load all data from an experiment directory."""
    exp_dir = Path(exp_dir)
    data = {'path': str(exp_dir)}
    
    # Load config
    config_path = exp_dir / 'config.json'
    if config_path.exists():
        with open(config_path) as f:
            data['config'] = json.load(f)
    
    # Load training history
    history_path = exp_dir / 'training_history.json'
    if history_path.exists():
        with open(history_path) as f:
            data['history'] = json.load(f)
    
    # Load AGOP metrics
    agop_path = exp_dir / 'agop_metrics.h5'
    if agop_path.exists():
        data['agop'] = {}
        with h5py.File(agop_path, 'r') as f:
            for key in f.keys():
                data['agop'][key] = f[key][:]
    
    # Load Lazy-Rich metrics
    lr_path = exp_dir / 'lazy_rich_metrics.h5'
    if lr_path.exists():
        data['lazy_rich'] = {}
        with h5py.File(lr_path, 'r') as f:
            for key in f.keys():
                data['lazy_rich'][key] = f[key][:]
    
    return data


def parse_experiment_name(exp_name):
    """Parse experiment name to extract configuration."""
    # Format: p{mod}_{attn}_{ln}_{opt}/wd{wd}_seed{seed}
    parts = exp_name.split('/')
    base_config = parts[0]
    wd_part = parts[1] if len(parts) > 1 else ''
    
    # Parse base config
    tokens = base_config.split('_')
    config = {
        'modulus': int(tokens[0][1:]),  # p97 -> 97
        'attention': tokens[1],
        'layernorm': tokens[2] == 'ln',
        'optimizer': tokens[3],
    }
    
    # Parse weight decay
    if wd_part.startswith('wd'):
        config['weight_decay'] = float(wd_part.split('_')[0][2:])
    
    return config


# Load all experiments
experiments = []

for base_dir in sorted(RESULTS_DIR.glob('p*_*_*_*')):
    if base_dir.is_dir():
        for wd_dir in sorted(base_dir.glob('wd*_seed*')):
            if wd_dir.is_dir():
                exp_name = f"{base_dir.name}/{wd_dir.name}"
                try:
                    data = load_experiment(wd_dir)
                    data['name'] = exp_name
                    data['parsed'] = parse_experiment_name(exp_name)
                    experiments.append(data)
                except Exception as e:
                    print(f"Error loading {exp_name}: {e}")

print(f"Loaded {len(experiments)} experiments")


## 2. Summary Statistics


In [ ]:
def detect_grokking(history, threshold=0.95):
    """Detect if/when grokking occurred."""
    test_accs = history.get('test_acc', [])
    epochs = history.get('epoch', [])
    
    for i, acc in enumerate(test_accs):
        if acc >= threshold:
            return True, epochs[i]
    return False, None


# Build summary DataFrame
summary_data = []

for exp in experiments:
    if 'history' not in exp:
        continue
    
    parsed = exp['parsed']
    history = exp['history']
    grokked, grok_epoch = detect_grokking(history)
    
    row = {
        'name': exp['name'],
        'modulus': parsed['modulus'],
        'attention': parsed['attention'],
        'layernorm': parsed['layernorm'],
        'optimizer': parsed['optimizer'],
        'weight_decay': parsed['weight_decay'],
        'final_train_acc': history['train_acc'][-1] if history['train_acc'] else None,
        'final_test_acc': history['test_acc'][-1] if history['test_acc'] else None,
        'grokked': grokked,
        'grok_epoch': grok_epoch,
    }
    
    # Add final AGOP metrics
    if 'agop' in exp:
        for key in ['agop_eigengap', 'agop_variation_collapse_ratio', 'agop_trace']:
            if key in exp['agop'] and len(exp['agop'][key]) > 0:
                row[f'final_{key}'] = exp['agop'][key][-1]
    
    # Add final NTK distance
    if 'lazy_rich' in exp and 'ntk_distance' in exp['lazy_rich']:
        if len(exp['lazy_rich']['ntk_distance']) > 0:
            row['final_ntk_distance'] = exp['lazy_rich']['ntk_distance'][-1]
            row['max_ntk_distance'] = max(exp['lazy_rich']['ntk_distance'])
    
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print(f"Summary DataFrame shape: {summary_df.shape}")
summary_df.head(10)


In [ ]:
# Grokking rate by configuration
print("="*60)
print("Grokking Rate by Factor")
print("="*60)

for factor in ['modulus', 'attention', 'layernorm', 'optimizer']:
    print(f"\n{factor.upper()}:")
    grok_rates = summary_df.groupby(factor)['grokked'].mean()
    for val, rate in grok_rates.items():
        n_grok = summary_df[summary_df[factor] == val]['grokked'].sum()
        n_total = (summary_df[factor] == val).sum()
        print(f"  {val}: {rate:.1%} ({n_grok}/{n_total})")


In [ ]:
# Grokking rate by weight decay
print("\nGROKKING RATE BY WEIGHT DECAY:")
wd_grok = summary_df.groupby('weight_decay').agg({
    'grokked': ['sum', 'count', 'mean'],
    'grok_epoch': 'mean'
})
wd_grok.columns = ['n_grokked', 'n_total', 'grok_rate', 'mean_grok_epoch']
print(wd_grok)


## 3. Training Curves Comparison


In [ ]:
def plot_training_curves_by_factor(experiments, factor, weight_decay=0.01, figsize=(14, 5)):
    """Plot training curves grouped by a specific factor."""
    
    # Filter experiments
    filtered = [e for e in experiments 
                if 'history' in e and 
                abs(e['parsed']['weight_decay'] - weight_decay) < 1e-10]
    
    if not filtered:
        print(f"No experiments found with weight_decay={weight_decay}")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Group by factor value
    groups = defaultdict(list)
    for exp in filtered:
        val = exp['parsed'][factor]
        groups[val].append(exp)
    
    colors = plt.cm.tab10.colors
    
    for idx, (val, exps) in enumerate(sorted(groups.items())):
        for exp in exps:
            history = exp['history']
            epochs = history['epoch']
            
            # Train accuracy
            axes[0].plot(epochs, history['train_acc'], 
                        color=colors[idx], alpha=0.3, linewidth=0.5)
            # Test accuracy
            axes[1].plot(epochs, history['test_acc'], 
                        color=colors[idx], alpha=0.3, linewidth=0.5)
        
        # Add legend entry
        axes[0].plot([], [], color=colors[idx], label=f'{factor}={val}', linewidth=2)
        axes[1].plot([], [], color=colors[idx], label=f'{factor}={val}', linewidth=2)
    
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Train Accuracy')
    axes[0].set_title(f'Training Accuracy by {factor.title()} (wd={weight_decay})')
    axes[0].legend()
    axes[0].set_ylim([0, 1.05])
    
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Test Accuracy')
    axes[1].set_title(f'Test Accuracy by {factor.title()} (wd={weight_decay})')
    axes[1].legend()
    axes[1].set_ylim([0, 1.05])
    
    plt.tight_layout()
    return fig

# Plot by each factor
for factor in ['attention', 'layernorm', 'optimizer', 'modulus']:
    fig = plot_training_curves_by_factor(experiments, factor, weight_decay=0.01)
    if fig:
        fig.savefig(FIGURES_DIR / f'training_curves_by_{factor}.png', dpi=150, bbox_inches='tight')
        plt.show()


## 4. AGOP Metrics Analysis


In [ ]:
def plot_agop_evolution(experiments, metric='agop_eigengap', n_cols=4, figsize=(16, 12)):
    """Plot AGOP metric evolution for all experiments."""
    
    # Filter experiments with AGOP data
    filtered = [e for e in experiments if 'agop' in e and metric in e['agop']]
    
    if not filtered:
        print(f"No experiments found with metric {metric}")
        return
    
    # Group by base config
    base_configs = {}
    for exp in filtered:
        p = exp['parsed']
        key = f"p{p['modulus']}_{p['attention']}_{('ln' if p['layernorm'] else 'noln')}_{p['optimizer']}"
        if key not in base_configs:
            base_configs[key] = []
        base_configs[key].append(exp)
    
    n_configs = len(base_configs)
    n_rows = (n_configs + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = axes.flatten()
    
    colors = plt.cm.viridis(np.linspace(0, 1, 6))  # 6 weight decays
    weight_decays = sorted(set(e['parsed']['weight_decay'] for e in filtered))
    wd_to_color = {wd: colors[i] for i, wd in enumerate(weight_decays)}
    
    for idx, (config, exps) in enumerate(sorted(base_configs.items())):
        ax = axes[idx]
        
        for exp in sorted(exps, key=lambda x: x['parsed']['weight_decay']):
            wd = exp['parsed']['weight_decay']
            epochs = exp['agop']['epoch']
            values = exp['agop'][metric]
            
            # Detect grokking
            grokked, grok_ep = detect_grokking(exp['history'])
            linestyle = '-' if grokked else '--'
            
            ax.plot(epochs, values, color=wd_to_color[wd], 
                   linestyle=linestyle, linewidth=1.5, 
                   label=f'wd={wd}' + (' *' if grokked else ''))
            
            if grokked and grok_ep is not None:
                ax.axvline(grok_ep, color=wd_to_color[wd], alpha=0.3, linestyle=':')
        
        ax.set_title(config, fontsize=9)
        ax.set_xlabel('Epoch', fontsize=8)
        ax.set_ylabel(metric.replace('agop_', ''), fontsize=8)
        ax.tick_params(labelsize=7)
        
        if idx == 0:
            ax.legend(fontsize=6, loc='upper right')
    
    # Hide empty subplots
    for idx in range(len(base_configs), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle(f'{metric.replace("_", " ").title()} Evolution\n(solid=grokked, dashed=not grokked)', 
                fontsize=12)
    plt.tight_layout()
    return fig

# Plot key AGOP metrics
for metric in ['agop_eigengap', 'agop_variation_collapse_ratio', 'agop_trace']:
    fig = plot_agop_evolution(experiments, metric)
    if fig:
        fig.savefig(FIGURES_DIR / f'{metric}_evolution.png', dpi=150, bbox_inches='tight')
        plt.show()


## 5. NTK Distance Analysis


In [ ]:
def plot_ntk_distance_evolution(experiments, n_cols=4, figsize=(16, 12)):
    """Plot NTK distance evolution for all experiments."""
    
    # Filter experiments with lazy-rich data
    filtered = [e for e in experiments 
                if 'lazy_rich' in e and 'ntk_distance' in e['lazy_rich']]
    
    if not filtered:
        print("No experiments found with NTK distance data")
        return
    
    # Group by base config
    base_configs = {}
    for exp in filtered:
        p = exp['parsed']
        key = f"p{p['modulus']}_{p['attention']}_{('ln' if p['layernorm'] else 'noln')}_{p['optimizer']}"
        if key not in base_configs:
            base_configs[key] = []
        base_configs[key].append(exp)
    
    n_configs = len(base_configs)
    n_rows = (n_configs + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = axes.flatten()
    
    colors = plt.cm.plasma(np.linspace(0, 1, 6))
    weight_decays = sorted(set(e['parsed']['weight_decay'] for e in filtered))
    wd_to_color = {wd: colors[i] for i, wd in enumerate(weight_decays)}
    
    for idx, (config, exps) in enumerate(sorted(base_configs.items())):
        ax = axes[idx]
        
        for exp in sorted(exps, key=lambda x: x['parsed']['weight_decay']):
            wd = exp['parsed']['weight_decay']
            epochs = exp['lazy_rich']['epoch']
            ntk_dist = exp['lazy_rich']['ntk_distance']
            
            grokked, grok_ep = detect_grokking(exp['history'])
            linestyle = '-' if grokked else '--'
            
            ax.semilogy(epochs, ntk_dist, color=wd_to_color[wd],
                       linestyle=linestyle, linewidth=1.5,
                       label=f'wd={wd}' + (' *' if grokked else ''))
            
            if grokked and grok_ep is not None:
                ax.axvline(grok_ep, color=wd_to_color[wd], alpha=0.3, linestyle=':')
        
        ax.set_title(config, fontsize=9)
        ax.set_xlabel('Epoch', fontsize=8)
        ax.set_ylabel('NTK Distance', fontsize=8)
        ax.tick_params(labelsize=7)
        
        if idx == 0:
            ax.legend(fontsize=6, loc='upper left')
    
    for idx in range(len(base_configs), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('NTK Distance Evolution (Lazy-to-Rich Transition)\n(solid=grokked, dashed=not grokked)',
                fontsize=12)
    plt.tight_layout()
    return fig

fig = plot_ntk_distance_evolution(experiments)
if fig:
    fig.savefig(FIGURES_DIR / 'ntk_distance_evolution.png', dpi=150, bbox_inches='tight')
    plt.show()


## 6. Grokking Heatmap by Configuration


In [ ]:
def plot_grokking_heatmap(summary_df):
    """Create heatmap showing grokking success across configurations."""
    
    # Create config string
    df = summary_df.copy()
    df['config'] = (df['attention'] + '_' + 
                   df['layernorm'].map({True: 'ln', False: 'noln'}) + '_' +
                   df['optimizer'])
    
    # Pivot for heatmap
    for modulus in [97, 113]:
        subset = df[df['modulus'] == modulus]
        
        pivot = subset.pivot_table(
            values='grokked',
            index='config',
            columns='weight_decay',
            aggfunc='mean'
        )
        
        plt.figure(figsize=(10, 6))
        sns.heatmap(pivot, annot=True, fmt='.0%', cmap='RdYlGn',
                   vmin=0, vmax=1, cbar_kws={'label': 'Grokking Rate'})
        plt.title(f'Grokking Rate by Configuration (Modulus p={modulus})')
        plt.xlabel('Weight Decay')
        plt.ylabel('Configuration (attention_layernorm_optimizer)')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f'grokking_heatmap_p{modulus}.png', dpi=150, bbox_inches='tight')
        plt.show()

if len(summary_df) > 0:
    plot_grokking_heatmap(summary_df)


## 7. Export Summary Statistics


In [ ]:
# Save summary to CSV
if len(summary_df) > 0:
    summary_df.to_csv(FIGURES_DIR / 'experiment_summary.csv', index=False)
    print(f"Saved summary to {FIGURES_DIR / 'experiment_summary.csv'}")
    
    # Print key statistics
    print("\n" + "="*60)
    print("KEY STATISTICS")
    print("="*60)
    print(f"Total experiments: {len(summary_df)}")
    print(f"Grokking rate: {summary_df['grokked'].mean():.1%}")
    grokked_df = summary_df[summary_df['grokked']]
    if len(grokked_df) > 0:
        print(f"Mean grokking epoch (when grokked): {grokked_df['grok_epoch'].mean():.0f}")

print("\nAnalysis complete!")
print(f"Figures saved to: {FIGURES_DIR.absolute()}")
